### Transformer

<img src="Architecture.png" width="500px">

#### Imports

In [2]:
import torch
import torch.nn as nn
import math

#### Input embedding layer

<img src="diagrams/InputLayer.png" width="1000px">

<img src="diagrams/InputEmbedding.png" width="1000px">

In [3]:
class InputEmbedding(nn.Module):
     def __init__(self, vocabulary_size: int, dimensionE: int) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.embedding = nn.Embedding(vocabulary_size, dimensionE)

     def forward(self, tokenIDs):
          embeddingStrength = math.sqrt(self.dimensionE)
          return self.embedding(tokenIDs) * embeddingStrength

In [4]:
tokenIDs = torch.tensor([2, 43, 21])
ie = InputEmbedding(vocabulary_size=200, dimensionE=6)(tokenIDs)
ie

tensor([[-3.6574,  3.7351,  0.6006, -5.4614, -3.1052, -0.0621],
        [-1.4211, -3.2588,  3.0808, -2.9143,  0.2613, -4.4190],
        [-1.8343,  1.5200, -1.6095, -2.1147,  3.3002, -2.2602]],
       grad_fn=<MulBackward0>)

#### Positional encoding

<img src="diagrams/PositionalEncoding.png" width="1000px">

In [5]:
class PositionalEncoding(nn.Module):
     def __init__(self, sequenceLength: int ,dimensionE: int, dropout: float) -> None:
          super().__init__()

          self.sequenceLength = sequenceLength
          self.dimensionE = dimensionE
          self.dropout = nn.Dropout(dropout)

          peMatrix = torch.zeros(sequenceLength, dimensionE)
          posVector = torch.arange(0, sequenceLength, dtype=torch.float).unsqueeze(1)
          frequencyCreator = torch.exp(torch.arange(0, dimensionE, 2).float() * (-math.log(10000.0) / dimensionE))

          peMatrix[:, 0::2] = torch.sin(posVector * frequencyCreator)
          peMatrix[:, 1::2] = torch.cos(posVector * frequencyCreator)

          peMatrix = peMatrix.unsqueeze(0)    # (1, sequenceLength, dimensionE)

          self.register_buffer("peMatrix", peMatrix)

     def forward(self, inputEmbedding):
          inputEmbedding = inputEmbedding + self.peMatrix[: , :inputEmbedding.shape[1], :].requires_grad_(False)
          return self.dropout(inputEmbedding)

In [6]:
PositionalEncoding(sequenceLength=3, dimensionE=6, dropout=0.1)(ie.unsqueeze(0))

tensor([[[-4.0638,  5.2612,  0.0000, -4.9571, -3.4502,  1.0421],
         [-0.6440, -3.0205,  3.4746, -2.1282,  0.2927, -3.7989],
         [-1.0278,  1.2265, -1.6853, -1.2434,  3.6717, -1.4002]]],
       grad_fn=<MulBackward0>)

#### Multi Head Attention

<img src="diagrams/MultiHeadAttention1.png" width="1000px"> 

<img src="diagrams/MultiHeadAttention2.png" width="1000px">

In [7]:
class MultiHeadAttentionBlock(nn.Module):
     def __init__(self, heads: int, dropout: float, dimensionE: int) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.dropout = nn.Dropout(dropout)
          self.heads = heads
          assert dimensionE % heads == 0, "Embedding vector must be equally divided among attention heads"
          self.sizeHeadVector = dimensionE // heads
          self.wQ = nn.Linear(dimensionE, dimensionE, bias=False)
          self.wK = nn.Linear(dimensionE, dimensionE, bias=False)
          self.wV = nn.Linear(dimensionE, dimensionE, bias=False)
          self.wO = nn.Linear(dimensionE, dimensionE, bias=False)

     @staticmethod
     def attention(query, key, value, mask, dropout: nn.Dropout):
          dimensionKey = key.shape[-1]
          attentionValues = query @ key.transpose(-2, -1)
          attentionValues /= math.sqrt(dimensionKey)
          if mask is not None:
               attentionValues = attentionValues.masked_fill(mask==0, -1e9)
          attentionValues = attentionValues.softmax(dim=-1)
          if dropout is not None:
               attentionValues = dropout(attentionValues)
          return attentionValues @ value, attentionValues
     
     def forward(self, q, k, v, mask):
          # (batch, sequenceLength, embedding vector size)
          query = self.wQ(q)         
          key = self.wK(k)
          value = self.wV(v)

          #  (batch size, sequenceLength, heads, embedding vector size)     to     (batch size, heads, sequenceLength, embedding vector size)
          query = query.view(query.shape[0], query.shape[1], self.heads, self.sizeHeadVector)      
          query = query.transpose(1, 2)            

          key = key.view(key.shape[0], key.shape[1], self.heads, self.sizeHeadVector)      
          key = key.transpose(1, 2)

          value = value.view(value.shape[0], value.shape[1], self.heads, self.sizeHeadVector)      
          value = value.transpose(1, 2)      

          attentionQKV, attentionValues = MultiHeadAttentionBlock.attention(
               query = query,
               key = key,
               value = value,
               mask = mask,
               dropout = self.dropout
          )

          attentionQKV = attentionQKV.transpose(1,2)
          attentionQKV = attentionQKV.contiguous()
          attentionQKV = attentionQKV.view(attentionQKV.shape[0], -1, self.heads * self.sizeHeadVector)      # batch, sequenceLength, dimensionE

          return self.wO(attentionQKV)

#### Layer Normalisation

In [8]:
class LayerNormalisation(nn.Module):
     def __init__(self, dimensionE: int, microValue: float = 10**-7) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.microValue = microValue
          self.gamma = nn.Parameter(torch.ones(dimensionE))
          self.beta = nn.Parameter(torch.zeros(dimensionE))

     def forward(self, input):
          mean = input.mean(dim=-1, keepdim=True)
          std = input.std(dim=-1, keepdim=True)
          normalisedInput = (input - mean) / (std + self.microValue)

          return self.gamma * normalisedInput + self.beta

#### Feed forward block

In [9]:
class FeedForwardBlock(nn.Module):
     def __init__(self, dimensionE: int, neurons: int, dropout: float) -> None:
          super().__init__()

          self.dimensionE = dimensionE
          self.neurons = neurons
          self.layer1 = nn.Linear(dimensionE, neurons)
          self.dropout = nn.Dropout(dropout)
          self.layer2 = nn.Linear(neurons, dimensionE)

     def forward(self, input):
          return self.layer2(self.dropout(torch.relu((self.layer1(input)))))

#### Residual Connection

In [10]:
class ResidualConnection(nn.Module):
     def __init__(self, dimensionE: int, dropout: float) -> None:
          super().__init__()
          self.dropout = nn.Dropout(dropout)
          self.normalise = LayerNormalisation(dimensionE)

     def forward(self, input, sublayer):
          connection = self.dropout(sublayer(self.normalise(input))) + input
          return connection

#### Encoder block

In [11]:
class EncoderBlock(nn.Module):
     def __init__(self, dimensionE: int, multiHeadAttention: MultiHeadAttentionBlock, feedForward: FeedForwardBlock) -> None:
          super().__init__()

          self.multiHeadAttention = multiHeadAttention
          self.feedForward = feedForward
          self.residualConnections = nn.ModuleList([ResidualConnection(dimensionE, dropout) for _ in range(2)])

     def forward(self, input, mask):
          input = self.residualConnections[0](input, lambda input: self.multiHeadAttention(input, input, input, mask))
          input = self.residualConnections[1](input, self.feedForward)
          return input

#### Encoder

In [12]:
class Encoder(nn.Module):
     def __init__(self, dimensionE: int, layers: nn.ModuleList) -> None:
          super().__init__()
          self.dimensionE = dimensionE
          self.layers = layers
          self.normalise = LayerNormalisation(dimensionE)

     def forward(self, input, mask):
          for layer in self.layers:
               input = layer(input, mask)
          return self.normalise(input)

#### Decoder block

In [ ]:
class DecoderBlock(nn.Module):
     def __init__(self, dimensionE: int, maskedMultiHeadAttention: MultiHeadAttentionBlock, multiHeadAttention: MultiHeadAttentionBlock, feedForward: FeedForwardBlock, dropout: float) -> None:
          super().__init__()

          self.maskedMultiHeadAttention = maskedMultiHeadAttention
          self.multiHeadAttention = multiHeadAttention
          self.feedForward = feedForward
          self.residualConnections = nn.ModuleList([ResidualConnection(dimensionE, dropout) for _ in range(3)])

     def forward(self, input, encoderOutput, sourceMask, targetMask):
          # source mask - hide tokens like <pad>
          # target mask - hide future tokens
          input = self.residualConnections[0](input, lambda input: self.maskedMultiHeadAttention(input, input, input, targetMask))
          input = self.residualConnections[1](input, lambda input: self.multiHeadAttention(input, encoderOutput, encoderOutput, sourceMask))
          input = self.residualConnections[2](input, self.feedForward)

          return input

SyntaxError: invalid syntax (3720865861.py, line 1)